In [1]:
import sys
import os
sys.path.append(os.path.abspath('../'))

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.models import PharmacokineticUDE
from src.data_gen import generate_dataset

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 12, "figure.dpi": 120})

In [ ]:
print("[*] Levantando el modelo entrenado...")

#Se instancia la arquitectura vacía
model = PharmacokineticUDE(hidden_dim=16)

#Se cargan los pesos descubiertos por train.py
model_path = '../checkpoints/ude_model_final.pth'
model.load_state_dict(torch.load(model_path))

#Se evalúa el modelo
model.eval()

#Se carga el ruido que el modelo descubrió
noise_path = '../checkpoints/noise_params_final.pth'
noise_params = torch.load(noise_path)

#Se imprimen los parámetros de la física que descubrió
kB_descubierto = model.ude_field.k_B.item()
print(f"-> Tasa de eliminación (k_B) aprendida: {kB_descubierto:.4f} (Real aprox: 0.2000)")

s_add_A = F.softplus(noise_params['add_A']).item()
s_prop_A = F.softplus(noise_params['prop_A']).item()
print(f"-> Ruido Droga A estimado | Aditivo: {s_add_A:.3f}, Proporcional: {s_prop_A:.3f}")

In [ ]:
#Se cargan los datos de entrenamiento
df_train = pd.read_csv('../data/synthetic/ddi_synthetic_data.csv')
t_train = torch.tensor(df_train['time'].values, dtype=torch.float32)
u0_train = torch.tensor([df_train['obs_A'].iloc[0], df_train['obs_B'].iloc[0]], dtype=torch.float32)

#Se realizan las predicciones
with torch.no_grad():
    pred_train = model(u0_train, t_train).numpy()

#Gráfico
fig, ax = plt.subplots(figsize=(8, 4))
t_np = t_train.numpy()

ax.scatter(t_np, df_train['obs_A'], color='red', alpha=0.4, label='Datos Entrenamiento A')
ax.plot(t_np, pred_train[:, 0], color='darkred', linewidth=2, label='Ajuste UDE A')

ax.scatter(t_np, df_train['obs_B'], color='blue', alpha=0.4, label='Datos Entrenamiento B')
ax.plot(t_np, pred_train[:, 1], color='darkblue', linewidth=2, label='Ajuste UDE B')

ax.set_title("Sanity Check: Ajuste sobre los datos de entrenamiento")
ax.legend()
plt.show()

In [ ]:
print("[*] Generando paciente de prueba con sobredosis de A (A=250) y poco inhibidor (B=50)")
#Se utiliza data_gen para generar la verdad de este nuevo paciente
df_test = generate_dataset(A0=250.0, B0=50.0, save_to_csv=False)

#Se extrae el tiempo y la nueva condición inicial
t_test = torch.tensor(df_test['time'].values, dtype=torch.float32)
u0_test = torch.tensor([250.0, 50.0], dtype=torch.float32)

#Se realiza la predicción
with torch.no_grad():
    pred_test = model(u0_test, t_test).numpy()

#Gráfico
fig, ax = plt.subplots(figsize=(10, 6))
t_np = t_test.numpy()

#Verdad del paciente
ax.plot(t_np, df_test['true_A'], '--', color='salmon', linewidth=3, label='Verdad Oculta A (No vista)')
ax.plot(t_np, df_test['true_B'], '--', color='cornflowerblue', linewidth=3, label='Verdad Oculta B (No vista)')

#Predicción de la UDE para el paciente
ax.plot(t_np, pred_test[:, 0], color='darkred', linewidth=2, label='Predicción UDE A (Red Neuronal)')
ax.plot(t_np, pred_test[:, 1], color='darkblue', linewidth=2, label='Predicción UDE B (Física)')

ax.set_title("Validación Fuera de Distribución: Extrapolación a Nuevas Dosis", pad=15)
ax.set_xlabel("Tiempo (horas)")
ax.set_ylabel("Concentración (mg/L)")
ax.legend(loc='upper right', framealpha=0.9)
plt.show()